![Hero — práctica 01](figs/hero_setup.png)

# Práctica del Día 01 (Tema 0) — Puesta a punto y primer contacto con un agente

**Asignatura:** Deep Learning para Business Analytics — Comillas (ICADE)
**Profesor:** Eduardo C. Garrido-Merchán · `ecgarrido@comillas.edu`

---

Esta práctica no es un examen. Es la prueba de que tu equipo está listo para todo el curso. **Sin esto aceptado no se entra en los kahoots ni en los casos prácticos.**

Cinco bloques:

1. Inventario del entorno Python (qué tenemos ya instalado).
2. Inventario de Ollama (qué modelos pesan cuánto, qué consumen en RAM).
3. Tres prompts con medida de latencia y tokens/segundo.
4. Dataset placeholder con su huella exacta en memoria.
5. Reflexión personal (indelegable).

**Política de IA.** Está permitida y fomentada. La única sección indelegable es la 5 (reflexión personal).

**Entrega.** El notebook ejecutado, un PDF exportado y un `README_DECLARACION_IA.md` con la herramienta o herramientas usadas y para qué.

## El pipeline completo que vamos a levantar

Antes de ejecutar nada, mira el coste real del stack en disco y los pasos por orden.

![Pipeline de instalación](figs/install_pipeline.png)

El cuello de botella es el modelo: ~2 GB para `qwen2.5:3b`. El resto del stack suma menos de medio giga.

## 1. Entorno Python

La librería `rich` nos permite mostrar tablas y paneles con la paleta del curso. Si no la tenéis, instaladla:

```bash
pip install rich psutil
```

Luego ejecutamos la celda siguiente y vemos qué tenemos.

In [ ]:
import platform
import sys

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.theme import Theme

DIGNUM = Theme({
    "ink": "bold #1A1A1A",
    "gold": "bold #B8860B",
    "golddeep": "bold #8C6508",
    "mute": "#8A8A8A",
    "warn": "bold #C36A1A",
    "err": "bold red",
    "step": "bold #8C6508",
})
con = Console(theme=DIGNUM)

rows = [
    ("Python", sys.version.split()[0], "gold"),
    ("OS", f"{platform.system()} {platform.release()}", "mute"),
]
for pkg in ("numpy", "pandas", "matplotlib", "torch", "transformers"):
    try:
        mod = __import__(pkg)
        rows.append((pkg, getattr(mod, "__version__", "?"), "gold"))
    except ImportError:
        rows.append((pkg, "no instalado", "warn"))

t = Table(title="Paquetes detectados", title_style="ink",
          header_style="golddeep", border_style="mute")
t.add_column("Componente", style="ink")
t.add_column("Versión / estado")
for name, val, role in rows:
    t.add_row(f"[{role}]{name}[/]", f"[{role}]{val}[/]")
con.print(t)
con.print(Panel.fit(
    "Si falta PyTorch o transformers, tres opciones gratuitas: Colab (T4), "
    "Kaggle (P100), Hugging Face Spaces.",
    title="[gold]Idea clave[/]", border_style="gold"))

**Pregunta 1.1.** Si tu equipo no tiene GPU, ¿qué plataforma usarías para entrenar un modelo de 30 MB en menos de cinco minutos? Razona la elección.

## 2. Despliegue local con Ollama

Instala Ollama desde <https://ollama.com/download>. En el terminal:

```bash
ollama serve                  # API en http://localhost:11434
ollama pull qwen2.5:3b        # ~2 GB en disco, ~2 GB en RAM
ollama pull qwen2.5-coder:7b  # ~5 GB, para código
```

El gráfico siguiente te indica cuánta RAM va a pedir cada tamaño de modelo y a qué velocidad responderá en CPU. Si tu portátil tiene 8 GB de RAM, evita los modelos de 7B y más.

![Memoria de cada modelo Ollama](figs/ollama_memory.png)

In [ ]:
import json
import urllib.error
import urllib.request


def fmt_gb(n_bytes):
    return f"{n_bytes / 2**30:.2f} GB"


def ollama_models():
    try:
        with urllib.request.urlopen(
                "http://localhost:11434/api/tags", timeout=5) as r:
            return json.loads(r.read().decode("utf-8")).get("models", [])
    except Exception:  # noqa: BLE001
        return None


models = ollama_models()
if models is None:
    con.print(Panel("Ollama no responde en localhost:11434. Arranca con "
                     "[gold]ollama serve[/] y vuelve a ejecutar.",
                     title="[err]Ollama no disponible[/]",
                     border_style="err"))
elif not models:
    con.print(Panel("No tienes modelos descargados todavía.\n"
                     "Lanza en una terminal aparte:\n"
                     "  [gold]ollama pull qwen2.5:3b[/]",
                     title="[warn]Sin modelos[/]",
                     border_style="warn"))
else:
    t = Table(title="Modelos instalados en Ollama",
              title_style="ink", header_style="golddeep",
              border_style="mute")
    t.add_column("Nombre", style="ink")
    t.add_column("Tamaño", style="gold", justify="right")
    t.add_column("Familia", style="mute")
    t.add_column("Parámetros", style="mute", justify="right")
    for m in models:
        d = m.get("details", {}) or {}
        t.add_row(m.get("name", "?"), fmt_gb(m.get("size", 0)),
                   d.get("family", "?"), d.get("parameter_size", "?"))
    con.print(t)

In [ ]:
import time

OLLAMA_CHAT = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:3b"


def chat_ollama(prompt, system="", model=MODEL):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    payload = {"model": model, "messages": messages, "stream": False,
               "options": {"temperature": 0.2}}
    req = urllib.request.Request(
        OLLAMA_CHAT,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"})
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=180) as r:
        data = json.loads(r.read().decode("utf-8"))
    t1 = time.perf_counter()
    tokens = data.get("eval_count", 0)
    eval_ns = data.get("eval_duration", 1)
    return {
        "content": data["message"]["content"],
        "latency_s": t1 - t0,
        "tokens_out": tokens,
        "tps": tokens / (eval_ns / 1e9) if eval_ns else 0.0,
    }


try:
    r = chat_ollama(
        "En una frase, explica por qué los Transformers han reemplazado a las RNN.",
        system="Eres profesor de Deep Learning para Business Analytics. Sé breve.",
    )
    con.print(Panel(
        r["content"],
        title=f"[gold]Respuesta del modelo[/]   "
              f"[mute]({r['latency_s']:.1f}s · {r['tps']:.1f} tok/s)[/]",
        border_style="gold"))
except Exception as e:  # noqa: BLE001
    con.print(f"[err]Ollama no responde[/]: {e}")

## 3. Tres prompts contra el mismo modelo

Vamos a lanzar tres consultas distintas y comparar latencia y tokens/segundo.

In [ ]:
triples = [
    ("Técnica",
     "Explica la diferencia entre regresión y clasificación con un ejemplo de retail.",
     "Eres profesor de Deep Learning para Business Analytics."),
    ("Negocio",
     "¿Cuál es el ROI típico de implantar un asistente RAG corporativo sobre 5.000 documentos legales?",
     "Eres consultor senior de IA."),
    ("Creativa",
     "Escribe un nombre para una startup que combina visión por computador y retail. Solo nombre y tagline.",
     "Eres brand-naming creativo."),
]

t = Table(title="Tres prompts contra el mismo modelo",
          title_style="ink", header_style="golddeep",
          border_style="mute", show_lines=True)
t.add_column("Tipo", style="ink", no_wrap=True)
t.add_column("Respuesta", style="ink", overflow="fold")
t.add_column("Lat.", justify="right", style="mute")
t.add_column("tok/s", justify="right", style="gold")
for label, prompt, system in triples:
    try:
        r = chat_ollama(prompt, system=system)
        short = (r["content"][:240] + "…") if len(r["content"]) > 240 else r["content"]
        t.add_row(label, short, f"{r['latency_s']:.1f}s",
                   f"{r['tps']:.1f}")
    except Exception as e:  # noqa: BLE001
        t.add_row(label, f"[err]ERROR {e}[/]", "—", "—")
con.print(t)

**Ejercicio 3.1.** Diseña tú **dos prompts más** (uno orientado a tu sector favorito, otro deliberadamente ambiguo). ¿La latencia depende más del prompt o de la complejidad de la respuesta? Justifica con números de la tabla.

## 4. Dataset placeholder con huella en memoria

Antes de elegir tu dataset definitivo, mira el catálogo recomendado del curso.

![Mosaico de datasets recomendados](figs/dataset_zoo.png)

Y de momento generamos un dataset sintético para que tengáis algo que tocar.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 200_000
df = pd.DataFrame({
    "producto": rng.choice(["A", "B", "C", "D"], size=n),
    "precio":   rng.uniform(5, 200, size=n).round(2),
    "unidades": rng.poisson(3, size=n),
})

t = Table(title=f"DataFrame placeholder · {n:,} filas",
          title_style="ink", header_style="golddeep",
          border_style="mute")
t.add_column("Columna", style="ink")
t.add_column("dtype", style="mute")
t.add_column("Mem (MB)", style="gold", justify="right")
for col in df.columns:
    t.add_row(col, str(df[col].dtype),
               f"{df[col].memory_usage(deep=True)/1024**2:.2f}")
mem_total = df.memory_usage(deep=True).sum() / 1024**2
t.add_row("[golddeep]TOTAL[/]", "", f"[golddeep]{mem_total:.2f}[/]")
con.print(t)
df.head()

**Ejercicio 4.1.** Sustituye el placeholder por un dataset real del repositorio del curso (`materiales/datasets/README.md`). Reporta, con la misma tabla que la celda anterior:

- Número de filas, columnas y dtype de cada columna.
- Huella en MB total y por columna.
- Una pregunta de negocio respondible con esos datos, un KPI medible asociado, y la familia de modelos candidata.

## 5. Agente CLI sobre este propio notebook

Llegados aquí ya tienes un LLM hablando contigo. Lo siguiente es entregarle un objetivo y dejarle actuar sobre tus ficheros.

![Bucle del agente](figs/agent_loop.png)

Dos pipelines posibles para esta sesión:

- **Pipeline A (cloud):** `npm install -g @anthropic-ai/claude-code` y luego `claude` en el directorio del notebook.
- **Pipeline B (libre):** `curl -fsSL https://opencode.ai/install | bash` y luego `opencode --model ollama/qwen2.5-coder:7b`.

Lanza el agente con este mensaje literal:

> *"Lee mi fichero `practica_01.ipynb`. Identifica las tres celdas marcadas como ejercicio. Para cada una, propón una mejora concreta que un alumno de Business Analytics agradecería. Devuelve una lista numerada y no edites todavía el fichero."*

Pega la respuesta del agente en la celda siguiente.

_Pipeline elegido:_

_Modelo:_

_Respuesta literal del agente:_

```
…
```

## 6. Reflexión de cierre (indelegable)

Estas tres preguntas no las puede contestar un LLM por ti. Son sobre tu propio uso.

1. ¿Qué encontraste más fácil de lo que esperabas y qué encontraste más difícil?
2. Con 100 € al mes de presupuesto, ¿API cloud o GPU local con modelos abiertos? Justifica.
3. Plantea **un caso de negocio** que te gustaría resolver durante el semestre. Una frase para problema, una para KPI, una para arquitectura candidata.

_Respuesta 1:_

_Respuesta 2:_

_Respuesta 3:_

## 7. Declaración obligatoria de uso de IA

| Sección | Herramienta | Para qué | Edición posterior |
|---|---|---|---|
| Sección 2 | Claude / Qwen / Ninguna | … | Sí / No |
| Sección 3 | … | … | … |
| Sección 4 | … | … | … |
| Sección 5 | Ninguna | — | — |
| Sección 6 | Ninguna | — | — |

Si no rellenas esta tabla, la práctica se devuelve sin corregir.